## Gaming Dataset Generator

Generates a synthetic **gaming** dataset in Unity Catalog with **realistic statistical distributions** and Faker-generated PII.

### Parameters

| Parameter | Default | Description |
| --- | --- | --- |
| `catalog` | `industry_sample_data` | Target Unity Catalog |

### Tables

| Schema | Entity Table | Rows | Event Table | Rows | Key Features |
| --- | --- | --- | --- | --- | --- |
| `gaming` | `players` | ~5K | `wager_records` | 100K-500K | Activity-earned VIP tiers, session tracking, behavior-driven RG flags |

### How to Run

1. Set the `catalog` widget parameter at the top of the notebook.
2. **Run All** — the notebook creates the `gaming` schema and generates both tables.
3. Final cells apply column comments and a `RemoveAfter` tag for workspace retention compliance.


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog);

DROP SCHEMA IF EXISTS IDENTIFIER(:catalog || '.gaming') CASCADE;

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.gaming');

In [0]:
%pip install faker --quiet

In [0]:
%restart_python

In [0]:
import random
import math
from datetime import datetime, timedelta, date
from faker import Faker
from pyspark.sql import Row

fake = Faker()
Faker.seed(88)
random.seed(88)
CATALOG = dbutils.widgets.get('catalog')
SCHEMA = "gaming"
CATALOG_SCHEMA = f"{CATALOG}.{SCHEMA}"
NOW = datetime(2026, 3, 21)

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

# --- Players table (~5,000 rows) ---
states = ["New Jersey", "Pennsylvania", "Michigan", "Connecticut", "West Virginia",
          "Delaware", "Nevada", "Colorado", "Arizona", "New York", "Illinois",
          "Louisiana", "Indiana", "Iowa", "Virginia", "Ohio", "Maryland", "Massachusetts",
          "Tennessee", "Kansas", "Kentucky", "Wyoming", "Maine", "Rhode Island"]
account_statuses = ["Active", "Dormant", "Suspended", "Self-Excluded", "Closed"]
verification_statuses = ["Verified", "Pending", "Unverified"]
vip_tiers = ["Bronze", "Silver", "Gold", "Platinum", "Diamond"]
preferred_games = ["Slots", "Sports Betting", "Blackjack", "Poker", "Roulette", "Live Dealer", "Baccarat"]
registration_sources = ["Organic Search", "Paid Ad", "Affiliate", "Referral", "Social Media", "TV Ad"]

NUM_PLAYERS = 5000
players = []
# Track wager totals per player for reconciliation
player_wager_totals = {}  # {id: (total_wagered, total_payout)}

for i in range(1, NUM_PLAYERS + 1):
    reg_date = date(2020, 1, 1) + timedelta(days=random.randint(0, 2000))
    age = int(clamp(random.gauss(35, 10), 21, 75))
    dob = date(2026, 3, 20) - timedelta(days=int(age * 365.25) + random.randint(0, 364))

    # VIP tier earned from activity (correlated with lifetime deposits later)
    activity_score = random.lognormvariate(3.0, 1.5)
    if activity_score > 1000: vip = "Diamond"
    elif activity_score > 400: vip = "Platinum"
    elif activity_score > 150: vip = "Gold"
    elif activity_score > 50: vip = "Silver"
    else: vip = "Bronze"

    # Self-exclusion as temporal event
    is_self_excluded = random.random() < 0.05
    if is_self_excluded:
        se_date = reg_date + timedelta(days=random.randint(90, (NOW.date() - reg_date).days))
        account_status = "Self-Excluded"
    else:
        se_date = None
        account_status = random.choices(["Active", "Dormant", "Suspended", "Closed"], weights=[70, 15, 5, 10])[0]

    # Deposits/withdrawals will be reconciled after wager generation
    pref_game = random.choices(preferred_games, weights=[30, 25, 15, 12, 8, 6, 4])[0]
    dep_limit = float(round(clamp(random.lognormvariate(5.8, 0.8), 50, 10000), 2)) if random.random() < 0.4 else None
    loss_limit = float(round(clamp(random.lognormvariate(5.2, 0.7), 25, 5000), 2)) if random.random() < 0.25 else None
    player_wager_totals[i] = [0.0, 0.0]  # [total_wagered, total_payout]

    players.append(Row(
        player_id=i,
        player_name=fake.name(),
        email=fake.email(),
        date_of_birth=dob,
        age=age,
        state=random.choice(states),
        registration_date=reg_date,
        registration_source=random.choices(registration_sources, weights=[20, 25, 25, 10, 12, 8])[0],
        account_status=account_status,
        verification_status=random.choices(verification_statuses, weights=[80, 12, 8])[0],
        vip_tier=vip,
        preferred_game=pref_game,
        deposit_limit_daily=dep_limit,
        loss_limit_daily=loss_limit,
        is_self_excluded=is_self_excluded,
        self_exclusion_date=se_date
    ))

print(f"Generated {len(players)} players (deposits/withdrawals reconciled after wager generation)")

# --- Wager Records table (randomized ~100K-500K rows) ---
game_catalog = {
    "Slots":          (["Mega Moolah", "Starburst", "Book of Dead", "Gonzo's Quest", "Divine Fortune",
                        "Buffalo Gold", "88 Fortunes", "Lightning Link"], 0.95),
    "Sports Betting": (["NFL Moneyline", "NBA Spread", "MLB Over/Under", "Soccer Parlay",
                        "Tennis Match", "UFC Fight", "NHL Puck Line", "Live In-Play"], 0.93),
    "Blackjack":      (["Classic Blackjack", "Blackjack Switch", "Spanish 21",
                        "Vegas Strip Blackjack", "Multi-Hand Blackjack"], 0.995),
    "Poker":          (["Texas Hold'em", "Omaha Hi-Lo", "Pot-Limit Omaha",
                        "Seven Card Stud", "Tournament MTT", "Sit & Go"], 0.97),
    "Roulette":       (["European Roulette", "American Roulette", "French Roulette",
                        "Lightning Roulette", "Auto Roulette"], 0.974),
    "Live Dealer":    (["Live Blackjack", "Live Roulette", "Live Baccarat",
                        "Dream Catcher", "Crazy Time", "Lightning Dice"], 0.975),
    "Baccarat":       (["Punto Banco", "Mini Baccarat", "Speed Baccarat",
                        "Commission Free Baccarat"], 0.986)
}
game_types = list(game_catalog.keys())
game_type_weights = [30, 25, 15, 12, 8, 6, 4]
platforms = ["Mobile App", "Desktop Web", "Mobile Web", "Tablet App"]
platform_weights = [45, 30, 18, 7]
wager_statuses = ["Settled", "Settled", "Settled", "Settled", "Voided", "Cashout"]
bonus_types = [None, None, None, None, None, "Free Spins", "Deposit Match", "Risk-Free Bet", "Loyalty Reward"]

wager_params = {
    "Slots":          (1.5, 1.2, 0.20, 500),
    "Sports Betting": (2.8, 1.3, 1, 10000),
    "Blackjack":      (3.2, 1.0, 5, 25000),
    "Poker":          (3.5, 1.2, 1, 50000),
    "Roulette":       (2.5, 1.0, 1, 5000),
    "Live Dealer":    (3.0, 0.9, 5, 10000),
    "Baccarat":       (3.8, 1.3, 10, 100000),
}

# Session tracking
session_counter = 0
current_sessions = {}  # player_id -> (session_id, wager_count)

records = []
NUM_EVENT_RECORDS = random.randint(100_000, 500_000)
for i in range(1, NUM_EVENT_RECORDS + 1):
    wager_ts = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 730),
                 hours=random.randint(0, 23), minutes=random.randint(0, 59))
    pid = random.randint(1, NUM_PLAYERS)
    game_type = random.choices(game_types, weights=game_type_weights)[0]
    games, base_rtp = game_catalog[game_type]
    game_name = random.choice(games)

    mu, sigma, lo, hi = wager_params[game_type]
    wager_amt = float(round(clamp(random.lognormvariate(mu, sigma), lo, hi), 2))

    outcome_roll = random.random()
    if outcome_roll < 0.45:
        raw_mult = 0.0
    elif outcome_roll < 0.68:
        raw_mult = clamp(random.betavariate(2, 3) * 1.5, 0.05, 0.95)
    elif outcome_roll < 0.82:
        raw_mult = clamp(random.gauss(1.1, 0.2), 0.80, 1.50)
    elif outcome_roll < 0.93:
        raw_mult = clamp(random.lognormvariate(0.7, 0.35), 1.5, 5.0)
    elif outcome_roll < 0.99:
        raw_mult = clamp(random.lognormvariate(1.5, 0.4), 4.0, 20.0)
    else:
        raw_mult = clamp(random.lognormvariate(2.5, 0.5), 15.0, 50.0)
    payout = float(round(wager_amt * raw_mult * base_rtp, 2))
    net_result = float(round(payout - wager_amt, 2))

    # Track per-player totals for reconciliation
    player_wager_totals[pid][0] += wager_amt
    player_wager_totals[pid][1] += payout

    bonus = random.choice(bonus_types)
    bonus_amt = float(round(clamp(random.lognormvariate(1.5, 0.8), 0.50, 200.0), 2)) if bonus else 0.0

    # Session assignment: 60% chance to continue existing session
    if pid in current_sessions and random.random() < 0.6:
        sess_id, wcount = current_sessions[pid]
        current_sessions[pid] = (sess_id, wcount + 1)
    else:
        session_counter += 1
        sess_id = f"SES-{session_counter:07d}"
        current_sessions[pid] = (sess_id, 1)

    session_min = round(clamp(random.lognormvariate(2.8, 1.0), 1.0, 720.0), 1)
    status = random.choice(wager_statuses)

    # Behavior-driven responsible gaming flag
    rg_flag = False
    cumulative_loss = player_wager_totals[pid][0] - player_wager_totals[pid][1]
    if cumulative_loss > 5000 and random.random() < 0.08: rg_flag = True
    if wager_amt > 1000 and random.random() < 0.04: rg_flag = True
    if session_min > 300 and random.random() < 0.06: rg_flag = True
    if net_result < -500 and random.random() < 0.05: rg_flag = True

    records.append(Row(
        wager_id=40000 + i,
        player_id=pid,
        wager_date=wager_ts,
        game_type=game_type,
        game_name=game_name,
        platform=random.choices(platforms, weights=platform_weights)[0],
        wager_amount=wager_amt,
        payout_amount=payout,
        net_result=net_result,
        wager_status=status,
        bonus_type=bonus,
        bonus_amount=bonus_amt,
        session_id=sess_id,
        session_duration_minutes=session_min,
        is_live_bet=game_type in ["Sports Betting", "Live Dealer"] and random.random() < 0.4,
        responsible_gaming_flag=rg_flag
    ))

records_df = spark.createDataFrame(records)
records_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.wager_records")
print(f"✔ Created {CATALOG_SCHEMA}.wager_records ({records_df.count()} rows)")

# Now backfill player deposits/withdrawals from actual wager data
reconciled_players = []
for p in players:
    totals = player_wager_totals[p.player_id]
    total_wagered = totals[0]
    total_payout = totals[1]
    # Deposits: slightly more than total wagered (account for deposits not fully played)
    deposits = float(round(total_wagered * clamp(random.gauss(1.15, 0.1), 1.02, 1.50), 2))
    # Withdrawals: fraction of payouts actually withdrawn
    withdrawal_ratio = clamp(random.betavariate(3.0, 2.0), 0.3, 0.95)
    withdrawals = float(round(total_payout * withdrawal_ratio, 2))
    reconciled_players.append(Row(
        player_id=p.player_id,
        player_name=p.player_name,
        email=p.email,
        date_of_birth=p.date_of_birth,
        age=p.age,
        state=p.state,
        registration_date=p.registration_date,
        registration_source=p.registration_source,
        account_status=p.account_status,
        verification_status=p.verification_status,
        vip_tier=p.vip_tier,
        preferred_game=p.preferred_game,
        lifetime_deposits=deposits,
        lifetime_withdrawals=withdrawals,
        deposit_limit_daily=p.deposit_limit_daily,
        loss_limit_daily=p.loss_limit_daily,
        is_self_excluded=p.is_self_excluded,
        self_exclusion_date=p.self_exclusion_date
    ))

players_df = spark.createDataFrame(reconciled_players)
players_df.write.mode("overwrite").saveAsTable(f"{CATALOG_SCHEMA}.players")
print(f"✔ Created {CATALOG_SCHEMA}.players ({players_df.count()} rows)")

print("\n--- Players (sample) ---")
display(players_df.limit(5))
print("\n--- Wager Records (sample) ---")
display(records_df.limit(5))

In [0]:
CATALOG = dbutils.widgets.get('catalog')

def apply_comments(table_fqn, comments):
    for col, comment in comments.items():
        spark.sql(f"ALTER TABLE {table_fqn} ALTER COLUMN `{col}` COMMENT '{comment}'")
    print(f"\u2714 {table_fqn} \u2014 {len(comments)} column comments applied")

apply_comments(f"{CATALOG}.gaming.players", {
    "player_id":           "Unique identifier for the player",
    "player_name":         "Full name (generated via Faker)",
    "email":               "Player email (generated via Faker)",
    "date_of_birth":       "Date of birth (DateType). All players are 21+",
    "age":                 "Current age in years",
    "state":               "US state of residence (24 states with legal online gambling)",
    "registration_date":   "Date the player account was created (DateType)",
    "registration_source": "Acquisition channel",
    "account_status":      "Account status: Active, Dormant, Suspended, Self-Excluded, or Closed",
    "verification_status": "Identity verification: Verified, Pending, or Unverified",
    "vip_tier":            "VIP tier earned from activity level: Bronze, Silver, Gold, Platinum, or Diamond",
    "preferred_game":      "Most-played game type",
    "lifetime_deposits":   "Total deposits in USD. Reconciled: slightly exceeds total wagered amount",
    "lifetime_withdrawals":"Total withdrawals in USD. Reconciled: fraction of actual payouts received",
    "deposit_limit_daily": "Self-imposed daily deposit limit; NULL if not set",
    "loss_limit_daily":    "Self-imposed daily loss limit; NULL if not set",
    "is_self_excluded":    "Whether the player opted into self-exclusion (~5%)",
    "self_exclusion_date": "Date self-exclusion was activated (DateType). NULL if not self-excluded",
})

apply_comments(f"{CATALOG}.gaming.wager_records", {
    "wager_id":                "Unique identifier for the wager",
    "player_id":               "Foreign key referencing players.player_id",
    "wager_date":              "Timestamp of the wager (TimestampType)",
    "game_type":               "Game category: Slots, Sports Betting, Blackjack, Poker, Roulette, Live Dealer, or Baccarat",
    "game_name":               "Specific game or event name",
    "platform":                "Platform: Mobile App, Desktop Web, Mobile Web, or Tablet App",
    "wager_amount":            "Amount wagered in USD",
    "payout_amount":           "Amount returned to player. Scaled by game-specific RTP",
    "net_result":              "Player profit/loss: payout - wager. Negative = house wins",
    "wager_status":            "Outcome: Settled, Voided, or Cashout",
    "bonus_type":              "Bonus applied if any: Free Spins, Deposit Match, Risk-Free Bet, Loyalty Reward, or NULL",
    "bonus_amount":            "Bonus value in USD. 0.0 if no bonus",
    "session_id":              "Session identifier (SES-XXXXXXX). Groups sequential wagers by the same player",
    "session_duration_minutes":"Length of the gaming session in minutes",
    "is_live_bet":             "Whether this was a live/in-play bet",
    "responsible_gaming_flag": "Behavior-driven flag. Triggered by: cumulative loss >$5K, single wager >$1K, session >5hrs, or large single loss",
})

print(f"\n\u2705 All column comments applied for gaming schema")

In [0]:
%sql
COMMENT ON SCHEMA IDENTIFIER(:catalog || '.gaming') IS
'Gaming sample dataset with realistic statistical distributions and Faker-generated PII. Entity table: `players` (~5K rows). Event table: `wager_records` (100K-500K rows). Key features: Activity-earned VIP tiers, session tracking, behavior-driven RG flags.';

In [0]:
CATALOG = dbutils.widgets.get('catalog')
remove_after_value = "2026-12-31"

spark.sql(f"ALTER SCHEMA `{CATALOG}`.`gaming` SET TAGS ('RemoveAfter' = '{remove_after_value}')")
print(f"✔ RemoveAfter tag applied to gaming schema ({remove_after_value})")